# Simple ESG Carbon RAG (Teaching Notebook, with Vector DB + Local LLM Option)

This notebook is for **beginner students** learning how RAG works in practice.

We build a **simple RAG pipeline** to extract carbon-emissions-related information from ESG / sustainability reports for:
- Tesla
- Amazon
- Google
- Apple
- Meta

## Teaching goal
Students should understand the flow:
1. Parse report text
2. Split into chunks
3. Store chunks in a vector database
4. Retrieve relevant chunks for a question
5. Ask an LLM to extract structured answers from retrieved evidence


## Practical Context: Green Finance / ESG Research

Analysts often need to extract:
- Scope 1 / Scope 2 / Scope 3 emissions
- Total GHG emissions
- Emissions intensity
- Carbon reduction targets
- Reporting year and boundary notes

RAG helps because ESG reports are long and noisy.
Instead of sending the whole report to an LLM, we retrieve only the most relevant passages.


## Dependencies (Install Outside Notebook Recommended)

For classroom use, you can run:

```bash
pip install pypdf pandas chromadb sentence-transformers transformers torch openai python-dotenv requests nbformat nbclient ipykernel
```

This notebook assumes dependencies are already installed in the current Python environment.


## Data Folder (Expected)

```text
RAG/
  simple_rag_esg_carbon_teaching.ipynb
  data/
    esg_reports/
      amazon_2024_sustainability_report.pdf
      apple_2025_environmental_progress_report.pdf
      google_2025_environmental_report.pdf
      meta_2025_sustainability_report.pdf
      tesla_2024_impact_report_extended_from_official_pdf_mirror.md   # fallback text mirror
```

Note:
- In this environment, the Tesla PDF was blocked by Tesla's CDN (Akamai 403), so we use a text mirror of the official PDF (`r.jina.ai` mirror of the Tesla PDF content).
- The parser in this notebook supports both `.pdf` and `.md/.txt`.


In [1]:
from __future__ import annotations

from pathlib import Path
import os
import re
import json
import time
from datetime import datetime
from typing import Any, Dict, List

import pandas as pd
import requests
from pypdf import PdfReader
from dotenv import load_dotenv

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

load_dotenv()

DATA_DIR = Path("data/esg_reports")
OUTPUT_DIR = Path("outputs")
VECTOR_DB_DIR = Path("vector_db/chroma_esg")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
VECTOR_DB_DIR.mkdir(exist_ok=True, parents=True)

print("Data dir:", DATA_DIR.resolve())
print("Output dir:", OUTPUT_DIR.resolve())
print("Vector DB dir:", VECTOR_DB_DIR.resolve())


Data dir: C:\Users\Jinquan Ye\OneDrive - Duke University\Research\teaching\RAG\data\esg_reports
Output dir: C:\Users\Jinquan Ye\OneDrive - Duke University\Research\teaching\RAG\outputs
Vector DB dir: C:\Users\Jinquan Ye\OneDrive - Duke University\Research\teaching\RAG\vector_db\chroma_esg


## Optional Download Helper (for reproducibility)

This cell shows the URLs used in this test.
You can set `DOWNLOAD_MISSING = True` to download files automatically.

Important:
- Tesla direct PDF may return `403 Forbidden` from some networks.
- This helper therefore includes a **Tesla text mirror fallback** (`r.jina.ai`) for teaching/demo purposes.


In [2]:
DOWNLOAD_MISSING = False

SOURCE_URLS = {
    "amazon_2024_sustainability_report.pdf": "https://sustainability.aboutamazon.com/2024-amazon-sustainability-report.pdf",
    "apple_2025_environmental_progress_report.pdf": "https://www.apple.com/environment/pdf/Apple_Environmental_Progress_Report_2025.pdf",
    "google_2025_environmental_report.pdf": "https://www.gstatic.com/gumdrop/sustainability/google-2025-environmental-report.pdf",
    "meta_2025_sustainability_report.pdf": "https://sustainability.atmeta.com/wp-content/uploads/2025/08/Meta_2025-Sustainability-Report_.pdf",
    "tesla_2024_impact_report_extended_from_official_pdf_mirror.md": "https://r.jina.ai/http://www.tesla.com/ns_videos/2024-extended-version-tesla-impact-report.pdf",
}

if DOWNLOAD_MISSING:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    headers = {"User-Agent": "Mozilla/5.0"}
    for name, url in SOURCE_URLS.items():
        path = DATA_DIR / name
        if path.exists() and path.stat().st_size > 100_000:
            print("SKIP", name)
            continue
        print("Downloading", name)
        r = requests.get(url, headers=headers, timeout=300)
        r.raise_for_status()
        if name.endswith(".pdf"):
            path.write_bytes(r.content)
        else:
            path.write_text(r.text, encoding="utf-8")
        print("Saved", name, path.stat().st_size, "bytes")
else:
    print("DOWNLOAD_MISSING = False (using local files if present)")

with open(OUTPUT_DIR / "download_sources.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "generated_at": datetime.utcnow().isoformat() + "Z",
            "source_urls": SOURCE_URLS,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )
print("Saved source URL record to", OUTPUT_DIR / "download_sources.json")


DOWNLOAD_MISSING = False (using local files if present)
Saved source URL record to outputs\download_sources.json


C:\Users\Jinquan Ye\AppData\Local\Temp\ipykernel_32900\3297289320.py:33: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at": datetime.utcnow().isoformat() + "Z",


## Step 1. Load Documents (PDF + Text Mirror Support)

We keep parsing lightweight:
- `pypdf` for normal PDFs
- direct text loading for `.md/.txt`

For the Tesla mirror file, we remove the `r.jina.ai` header and keep the report content section.


In [3]:
def extract_text_from_pdf(pdf_path: Path) -> str:
    reader = PdfReader(str(pdf_path))
    parts = []
    for page in reader.pages:
        parts.append(page.extract_text() or "")
    return "\n".join(parts)


def extract_text_from_text_file(path: Path) -> str:
    text = path.read_text(encoding="utf-8", errors="ignore")
    # r.jina.ai PDF mirror format includes a header and then "Markdown Content:"
    marker = "Markdown Content:"
    if marker in text:
        text = text.split(marker, 1)[1].strip()
    return text


def infer_company_label(file_name: str) -> str:
    lower = file_name.lower()
    if lower.startswith("tesla_"):
        return "Tesla"
    if lower.startswith("amazon_"):
        return "Amazon"
    if lower.startswith("google_"):
        return "Google"
    if lower.startswith("apple_"):
        return "Apple"
    if lower.startswith("meta_"):
        return "Meta"
    return Path(file_name).stem


def load_documents(data_dir: Path) -> List[Dict[str, Any]]:
    docs: List[Dict[str, Any]] = []
    for path in sorted(data_dir.iterdir()):
        if path.suffix.lower() not in {".pdf", ".md", ".txt"}:
            continue
        try:
            if path.suffix.lower() == ".pdf":
                text = extract_text_from_pdf(path)
                source_type = "pdf"
            else:
                text = extract_text_from_text_file(path)
                source_type = "text_mirror"
        except Exception as e:
            print(f"SKIP (parse error): {path.name} -> {e}")
            continue

        docs.append(
            {
                "doc_id": path.stem,
                "company": infer_company_label(path.name),
                "file_name": path.name,
                "source_type": source_type,
                "text": text,
                "char_count": len(text),
            }
        )
    return docs


documents = load_documents(DATA_DIR) if DATA_DIR.exists() else []
print(f"Loaded {len(documents)} document(s).")
for d in documents:
    print(f"- {d['company']}: {d['file_name']} ({d['source_type']}) | {d['char_count']:,} chars")


invalid pdf header: b'<HTML'


EOF marker not found


SKIP (parse error): tesla_2024_impact_report_extended.pdf -> Stream has ended unexpectedly
Loaded 5 document(s).
- Amazon: amazon_2024_sustainability_report.pdf (pdf) | 308,698 chars
- Apple: apple_2025_environmental_progress_report.pdf (pdf) | 416,889 chars
- Google: google_2025_environmental_report.pdf (pdf) | 386,367 chars
- Meta: meta_2025_sustainability_report.pdf (pdf) | 116,585 chars
- Tesla: tesla_2024_impact_report_extended_from_official_pdf_mirror.md (text_mirror) | 48,423 chars


## Step 2. Chunk the Documents

We split report text into chunks before storing them in the vector database.

This is still a simple, teaching-friendly chunker:
- whitespace cleanup
- paragraph-aware chunking
- small overlap


In [4]:
def normalize_text(text: str) -> str:
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def chunk_text(text: str, max_chars: int = 900, overlap_chars: int = 120) -> List[str]:
    text = normalize_text(text)
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks: List[str] = []
    current = ""

    for para in paragraphs:
        candidate = f"{current}\n\n{para}".strip() if current else para
        if len(candidate) <= max_chars:
            current = candidate
            continue

        if current:
            chunks.append(current)
            current = ""

        if len(para) <= max_chars:
            current = para
        else:
            start = 0
            while start < len(para):
                end = start + max_chars
                piece = para[start:end]
                chunks.append(piece)
                if end >= len(para):
                    break
                start = max(0, end - overlap_chars)

    if current:
        chunks.append(current)

    # lightweight overlap between adjacent chunks
    if overlap_chars > 0 and len(chunks) > 1:
        out = [chunks[0]]
        for i in range(1, len(chunks)):
            prefix = chunks[i - 1][-overlap_chars:]
            out.append((prefix + "\n" + chunks[i]).strip())
        chunks = out

    return chunks


def build_chunk_records(docs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    records: List[Dict[str, Any]] = []
    for doc in docs:
        chunks = chunk_text(doc["text"])
        for i, chunk in enumerate(chunks):
            records.append(
                {
                    "id": f"{doc['doc_id']}::chunk::{i}",
                    "doc_id": doc["doc_id"],
                    "company": doc["company"],
                    "file_name": doc["file_name"],
                    "source_type": doc["source_type"],
                    "chunk_id": i,
                    "text": chunk,
                }
            )
    return records


chunk_records = build_chunk_records(documents)
print("Total chunks:", len(chunk_records))
if chunk_records:
    print("Example chunk:", chunk_records[0]["id"])
    print(chunk_records[0]["text"][:600])


Total chunks: 1648
Example chunk: amazon_2024_sustainability_report::chunk::0
2024
Amazon
Sustainability
Report


## Step 3. Build a Vector Database (ChromaDB)

Now we store chunks in a vector database using:
- `ChromaDB` (local persistent vector DB)
- `all-MiniLM-L6-v2` embeddings (small and good for teaching demos)


In [5]:
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
COLLECTION_NAME = "esg_carbon_chunks_demo_v2"
REBUILD_VECTOR_DB = True  # set False if you want to reuse an existing collection

embedding_fn = SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL_NAME)
chroma_client = chromadb.PersistentClient(path=str(VECTOR_DB_DIR))

if REBUILD_VECTOR_DB:
    try:
        chroma_client.delete_collection(COLLECTION_NAME)
        print("Deleted existing collection:", COLLECTION_NAME)
    except Exception:
        pass

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},
)


def index_chunks_to_chroma(collection, chunk_records: List[Dict[str, Any]], batch_size: int = 64) -> None:
    if not chunk_records:
        print("No chunks to index.")
        return

    for start in range(0, len(chunk_records), batch_size):
        batch = chunk_records[start : start + batch_size]
        collection.add(
            ids=[r["id"] for r in batch],
            documents=[r["text"] for r in batch],
            metadatas=[
                {
                    "doc_id": r["doc_id"],
                    "company": r["company"],
                    "file_name": r["file_name"],
                    "source_type": r["source_type"],
                    "chunk_id": int(r["chunk_id"]),
                }
                for r in batch
            ],
        )
    print("Indexed", len(chunk_records), "chunks into ChromaDB.")


index_chunks_to_chroma(collection, chunk_records)
print("Collection count:", collection.count())


C:\Users\Jinquan Ye\OneDrive - Duke University\Research\teaching\RAG\.venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\Jinquan Ye\OneDrive - Duke University\Research\teaching\RAG\.venv312\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Jinquan Ye\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Indexed 1648 chunks into ChromaDB.
Collection count: 1648


## Step 4. Retrieval from the Vector DB

This function retrieves the most relevant chunks for a question.
We can also filter to a single company/report using metadata (`doc_id`).


In [6]:
def retrieve_chunks(query: str, top_k: int = 4, filter_doc_id: str | None = None) -> List[Dict[str, Any]]:
    kwargs: Dict[str, Any] = {
        "query_texts": [query],
        "n_results": top_k,
        "include": ["documents", "metadatas", "distances"],
    }
    if filter_doc_id is not None:
        kwargs["where"] = {"doc_id": filter_doc_id}

    res = collection.query(**kwargs)

    docs = res.get("documents", [[]])[0]
    metas = res.get("metadatas", [[]])[0]
    distances = res.get("distances", [[]])[0]

    out = []
    for doc_text, meta, distance in zip(docs, metas, distances):
        item = dict(meta)
        item["text"] = doc_text
        item["distance"] = float(distance)
        # cosine distance -> rough similarity (teaching convenience)
        item["similarity"] = max(0.0, 1.0 - float(distance))
        out.append(item)
    return out


demo_query = "Scope 1 Scope 2 Scope 3 emissions total GHG emissions emissions intensity carbon targets reporting year"
if documents:
    demo_doc_id = documents[0]["doc_id"]
    hits = retrieve_chunks(demo_query, top_k=3, filter_doc_id=demo_doc_id)
    print("Demo retrieval for:", demo_doc_id)
    for i, h in enumerate(hits, 1):
        print(f"\n[{i}] {h['file_name']} | chunk={h['chunk_id']} | similarity={h['similarity']:.4f}")
        print(h["text"][:500])
else:
    print("No documents found.")


Demo retrieval for: amazon_2024_sustainability_report

[1] amazon_2024_sustainability_report.pdf | chunk=358 | similarity=0.5975
3,485
*Scope 2 and 3 carbon emissions are calculated using a market-based method. **This table includes both on-site sol
3,485
*Scope 2 and 3 carbon emissions are calculated using a market-based method. **This table includes both on-site solar and contracted off-site utility-scale wind and solar projects, which are in various stages of development and construction. Of the projects included in the table, 124 were announced 
in January 2025. AWS aims to procure renewable electricity in the same gr

[2] amazon_2024_sustainability_report.pdf | chunk=357 | similarity=0.5598
n Devices) 0.04 0.04 0.03 -25%
Amazon’s Carbon Footprint 51.17 60.64 71.54 65.10 64.38 68.25 6%
2022 and 2023 carbon foo
n Devices) 0.04 0.04 0.03 -25%
Amazon’s Carbon Footprint 51.17 60.64 71.54 65.10 64.38 68.25 6%
2022 and 2023 carbon footprints 
recalculated in accordance with 
updated 
Car

## Step 5. LLM Backends (Local Distilled Model / DeepSeek / ChatGPT)

We provide a **single extraction pipeline** and make the LLM backend swappable.

### Backend A: Local distilled model (default in this notebook)
- Suggested model: `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B`
- Runs locally (CPU is possible, but slower)

### Backend B: DeepSeek API
- Set `DEEPSEEK_API_KEY`

### Backend C: ChatGPT / OpenAI API
- Set `OPENAI_API_KEY`

Teaching point:
- Retrieval stays the same.
- Only the generation backend changes.


In [7]:
from openai import OpenAI

_LOCAL_LLM_CACHE: Dict[str, Any] = {}


def get_local_llm(model_id: str | None = None):
    """
    Load a local instruction model via transformers.
    Default uses a distilled DeepSeek-R1 Qwen model.
    """
    model_id = model_id or os.getenv("LOCAL_MODEL_ID", "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
    if model_id in _LOCAL_LLM_CACHE:
        return _LOCAL_LLM_CACHE[model_id]

    print(f"Loading local model: {model_id} (first load may take a while)")
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float32,  # CPU-safe default
    )
    model.eval()
    _LOCAL_LLM_CACHE[model_id] = (tokenizer, model)
    return tokenizer, model


def call_local_llm(system_prompt: str, user_prompt: str, model_id: str | None = None, max_new_tokens: int = 400) -> str:
    tokenizer, model = get_local_llm(model_id=model_id)
    import torch

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    if hasattr(tokenizer, "apply_chat_template"):
        prompt_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        # Generic fallback for older/non-chat tokenizers
        prompt_text = f"System:\\n{system_prompt}\\n\\nUser:\\n{user_prompt}\\n\\nAssistant:\\n"

    # Keep prompt length conservative for small local models
    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=3072,
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_tokens = outputs[0][inputs["input_ids"].shape[-1] :]
    return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()


def get_api_client(provider: str):
    provider = provider.lower().strip()
    if provider == "openai":
        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            raise ValueError("Missing OPENAI_API_KEY")
        return OpenAI(api_key=api_key), os.getenv("OPENAI_MODEL", "gpt-4o-mini")
    if provider == "deepseek":
        api_key = os.getenv("DEEPSEEK_API_KEY")
        if not api_key:
            raise ValueError("Missing DEEPSEEK_API_KEY")
        base_url = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com")
        return OpenAI(api_key=api_key, base_url=base_url), os.getenv("DEEPSEEK_MODEL", "deepseek-chat")
    raise ValueError("provider must be 'openai' or 'deepseek'")


def call_api_llm(provider: str, system_prompt: str, user_prompt: str, model: str | None = None) -> str:
    client, default_model = get_api_client(provider)
    model = model or default_model
    resp = client.chat.completions.create(
        model=model,
        temperature=0.0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return resp.choices[0].message.content or ""


def call_llm(provider: str, system_prompt: str, user_prompt: str, model: str | None = None) -> str:
    provider = provider.lower().strip()
    if provider == "local":
        return call_local_llm(system_prompt, user_prompt, model_id=model)
    if provider in {"openai", "deepseek"}:
        return call_api_llm(provider, system_prompt, user_prompt, model=model)
    raise ValueError("provider must be one of: local, openai, deepseek")


## Step 6. Build the Extraction Prompt

We ask the model to return JSON only.
This makes it easier to convert results into a table for analysis.


In [8]:
SYSTEM_PROMPT = """
You are an ESG research assistant.
Extract carbon-emissions-related information from report excerpts.
Use only the provided excerpts.
Return JSON only.
If a field is missing, use null.
Do not invent numbers.
""".strip()

EXTRACTION_SCHEMA = {
    "company": "string or null",
    "report_year": "string or null",
    "scope_1_emissions": "string or null",
    "scope_2_emissions": "string or null",
    "scope_3_emissions": "string or null",
    "total_ghg_emissions": "string or null",
    "emissions_intensity": "string or null",
    "target_summary": "string or null",
    "evidence_quotes": ["short quotes/snippets from retrieved excerpts"],
    "confidence": "low | medium | high",
    "notes": "ambiguity / unit / boundary notes",
}


def build_extraction_prompt(company_name: str, query: str, chunks: List[Dict[str, Any]]) -> str:
    blocks = []
    for i, c in enumerate(chunks, 1):
        blocks.append(
            f"[Chunk {i}] file={c['file_name']} chunk_id={c['chunk_id']} similarity={c['similarity']:.4f}\n{c['text']}"
        )

    return f"""
Task: Extract carbon emissions information for {company_name}.

Question:
{query}

Output JSON schema (use these exact keys):
{json.dumps(EXTRACTION_SCHEMA, indent=2)}

Retrieved report excerpts:
{chr(10).join([''] + [b + chr(10) for b in blocks])}

Rules:
1. Use only retrieved excerpts.
2. Preserve units as written.
3. If multiple years appear, choose the most likely reporting year and explain ambiguity in notes.
4. Return JSON only (no markdown).
""".strip()


In [9]:
def extract_first_json_object(text: str) -> str:
    text = text.strip()
    # Fast path
    if text.startswith("{") and text.endswith("}"):
        return text

    # Find first balanced JSON object
    start = text.find("{")
    if start == -1:
        raise ValueError("No JSON object start found")

    depth = 0
    for i in range(start, len(text)):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start : i + 1]
    raise ValueError("No balanced JSON object found")


def parse_model_json(raw_text: str) -> Dict[str, Any]:
    try:
        return json.loads(raw_text)
    except Exception:
        candidate = extract_first_json_object(raw_text)
        return json.loads(candidate)


## Step 7. Single-Company RAG Extraction Function

This function combines:
- vector retrieval (ChromaDB)
- prompt building
- LLM call (local or API)
- JSON parsing


In [10]:
RAG_QUERY = (
    "Find carbon emissions information including Scope 1, Scope 2, Scope 3, total GHG emissions, "
    "emissions intensity, reporting year, and emissions reduction targets."
)


def rag_extract_for_doc(
    doc: Dict[str, Any],
    provider: str = "local",
    model: str | None = None,
    top_k: int = 4,
) -> Dict[str, Any]:
    hits = retrieve_chunks(RAG_QUERY, top_k=top_k, filter_doc_id=doc["doc_id"])
    if not hits:
        return {"company": doc["company"], "error": f"No hits for {doc['doc_id']}"}

    user_prompt = build_extraction_prompt(doc["company"], RAG_QUERY, hits)
    t0 = time.time()
    raw = call_llm(provider=provider, system_prompt=SYSTEM_PROMPT, user_prompt=user_prompt, model=model)
    elapsed = time.time() - t0

    try:
        data = parse_model_json(raw)
    except Exception as e:
        data = {
            "company": doc["company"],
            "parse_error": str(e),
            "raw_model_output": raw[:5000],
        }

    data["_rag_meta"] = {
        "provider": provider,
        "model": model or ("LOCAL_MODEL_ID env / default" if provider == "local" else "provider default"),
        "doc_id": doc["doc_id"],
        "file_name": doc["file_name"],
        "source_type": doc["source_type"],
        "top_k": top_k,
        "latency_sec": round(elapsed, 2),
        "retrieved_chunks": [
            {
                "chunk_id": h["chunk_id"],
                "similarity": h["similarity"],
                "file_name": h["file_name"],
            }
            for h in hits
        ],
    }
    return data


## Step 8. Configure the Backend for This Run

Default settings below use a **local distilled model**.

If your CPU is slow or RAM is limited, you can set:
- `LOCAL_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"` (smaller fallback, not distilled)

API alternatives:
- `provider = "deepseek"`
- `provider = "openai"`


In [11]:
# Choose backend: "local", "deepseek", or "openai"
provider = "local"

# Local default: deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
# Optional smaller fallback example:
# model = "Qwen/Qwen2.5-0.5B-Instruct"
model = None

top_k = 4

print("provider =", provider)
print("model =", model)
print("documents =", [d["company"] for d in documents])


provider = local
model = None
documents = ['Amazon', 'Apple', 'Google', 'Meta', 'Tesla']


## Step 9. Test One Company First (Recommended)

Run one company first to confirm the pipeline works before batch mode.


In [12]:
RUN_SINGLE_TEST = True

single_result = None
if RUN_SINGLE_TEST and documents:
    # Start with Tesla if available (using mirrored text file in this environment)
    preferred_order = ["Tesla", "Amazon", "Google", "Apple", "Meta"]
    selected = None
    for name in preferred_order:
        selected = next((d for d in documents if d["company"] == name), None)
        if selected:
            break

    print("Running single-company test for:", selected["company"], "|", selected["file_name"])
    single_result = rag_extract_for_doc(selected, provider=provider, model=model, top_k=top_k)
    print(json.dumps(single_result, indent=2, ensure_ascii=False)[:4000])
else:
    print("Skipping single-company test.")


Running single-company test for: Tesla | tesla_2024_impact_report_extended_from_official_pdf_mirror.md
Loading local model: deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B (first load may take a while)


C:\Users\Jinquan Ye\OneDrive - Duke University\Research\teaching\RAG\.venv312\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Jinquan Ye\.cache\huggingface\hub\models--deepseek-ai--DeepSeek-R1-Distill-Qwen-1.5B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


`torch_dtype` is deprecated! Use `dtype` instead!


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{
  "company": "Tesla",
  "parse_error": "No JSON object start found",
  "raw_model_output": "Okay, so I need to extract carbon emissions information from the Tesla report excerpts. The user provided four chunks of the report, each with different sections. My task is to extract specific fields like Scope 1, Scope 2, Scope 3 emissions, total GHG emissions, emissions intensity, reporting year, and emissions reduction targets. I also need to return this information in JSON format without adding any numbers or explanations, just the JSON structure.\n\nFirst, I'll start by looking at each chunk to see what information is available. The first chunk mentions \"Management Assertion Scope 1 & 2 GHG Emissions\" and talks about impacts and data-driven approaches. The second chunk is more detailed, discussing various aspects like data collection, assessment criteria, and specific metrics like water withdrawal and vehicle safety waste. The third chunk seems to provide additional metrics and key met

## Step 10. Batch Run on Tesla / Amazon / Google / Apple / Meta

This cell runs the full pipeline across the five-company test set and saves outputs.


In [13]:
TARGET_COMPANIES = ["Tesla", "Amazon", "Google", "Apple", "Meta"]
RUN_BATCH = True


def batch_extract(
    docs: List[Dict[str, Any]],
    provider: str = "local",
    model: str | None = None,
    top_k: int = 4,
    target_companies: List[str] | None = None,
) -> List[Dict[str, Any]]:
    selected_docs = docs
    if target_companies:
        target_set = set(target_companies)
        selected_docs = [d for d in docs if d["company"] in target_set]

    results = []
    for doc in selected_docs:
        print(f"\n=== Processing {doc['company']} | {doc['file_name']} ===")
        try:
            result = rag_extract_for_doc(doc, provider=provider, model=model, top_k=top_k)
        except Exception as e:
            result = {
                "company": doc["company"],
                "error": str(e),
                "_rag_meta": {
                    "doc_id": doc["doc_id"],
                    "file_name": doc["file_name"],
                    "source_type": doc["source_type"],
                },
            }
        results.append(result)
    return results


all_results: List[Dict[str, Any]] = []
if RUN_BATCH:
    all_results = batch_extract(
        documents,
        provider=provider,
        model=model,
        top_k=top_k,
        target_companies=TARGET_COMPANIES,
    )
    print("\nBatch complete:", len(all_results), "result(s)")
else:
    print("RUN_BATCH = False")



=== Processing Amazon | amazon_2024_sustainability_report.pdf ===



=== Processing Apple | apple_2025_environmental_progress_report.pdf ===



=== Processing Google | google_2025_environmental_report.pdf ===



=== Processing Meta | meta_2025_sustainability_report.pdf ===



=== Processing Tesla | tesla_2024_impact_report_extended_from_official_pdf_mirror.md ===



Batch complete: 5 result(s)


## Step 11. Convert Results to a Table

This makes the extraction output easier to inspect and compare across firms.


In [14]:
summary_df = pd.DataFrame(
    [
        {
            "company": r.get("company"),
            "report_year": r.get("report_year"),
            "scope_1_emissions": r.get("scope_1_emissions"),
            "scope_2_emissions": r.get("scope_2_emissions"),
            "scope_3_emissions": r.get("scope_3_emissions"),
            "total_ghg_emissions": r.get("total_ghg_emissions"),
            "emissions_intensity": r.get("emissions_intensity"),
            "target_summary": r.get("target_summary"),
            "confidence": r.get("confidence"),
            "notes": r.get("notes"),
            "error": r.get("error"),
            "parse_error": r.get("parse_error"),
            "latency_sec": (r.get("_rag_meta") or {}).get("latency_sec"),
            "source_type": (r.get("_rag_meta") or {}).get("source_type"),
            "file_name": (r.get("_rag_meta") or {}).get("file_name"),
        }
        for r in all_results
    ]
)

summary_df


,company,report_year,scope_1_emissions,scope_2_emissions,scope_3_emissions,total_ghg_emissions,emissions_intensity,target_summary,confidence,notes,error,parse_error,latency_sec,source_type,file_name
0,Amazon,None,None,None,None,None,None,None,None,None,None,No balanced JSON object found,181.75,pdf,amazon_2024_sustainability_report.pdf
1,Apple,None,None,None,None,None,None,None,None,None,None,No JSON object start found,184.75,pdf,apple_2025_environmental_progress_report.pdf
2,Google,None,None,None,None,None,None,None,None,None,None,No JSON object start found,169.66,pdf,google_2025_environmental_report.pdf
3,Meta,None,None,None,None,None,None,None,None,None,None,No JSON object start found,180.84,pdf,meta_2025_sustainability_report.pdf
4,Tesla,None,None,None,None,None,None,None,None,None,None,No JSON object start found,173.43,text_mirror,tesla_2024_impact_report_extended_from_officia...


## Step 12. Save Outputs and Run Record

We save:
- JSON results
- CSV summary table
- a run-record JSON (backend, model, timestamp, files used)


In [15]:
run_timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")

results_json_path = OUTPUT_DIR / "esg_carbon_rag_results.json"
summary_csv_path = OUTPUT_DIR / "esg_carbon_rag_results.csv"
run_record_path = OUTPUT_DIR / f"notebook_run_record_{run_timestamp}.json"

with open(results_json_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

summary_df.to_csv(summary_csv_path, index=False)

run_record = {
    "run_timestamp_utc": run_timestamp,
    "provider": provider,
    "model": model,
    "top_k": top_k,
    "target_companies": TARGET_COMPANIES,
    "doc_count_loaded": len(documents),
    "doc_files": [d["file_name"] for d in documents],
    "vector_db_path": str(VECTOR_DB_DIR),
    "vector_collection": COLLECTION_NAME,
    "embedding_model": EMBED_MODEL_NAME,
    "results_json_path": str(results_json_path),
    "summary_csv_path": str(summary_csv_path),
}

with open(run_record_path, "w", encoding="utf-8") as f:
    json.dump(run_record, f, indent=2, ensure_ascii=False)

print("Saved:", results_json_path)
print("Saved:", summary_csv_path)
print("Saved:", run_record_path)


Saved: outputs\esg_carbon_rag_results.json
Saved: outputs\esg_carbon_rag_results.csv
Saved: outputs\notebook_run_record_20260225T100221Z.json


C:\Users\Jinquan Ye\AppData\Local\Temp\ipykernel_32900\514416829.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  run_timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")


## Discussion Prompts for Students

1. Which company was easiest/hardest to extract and why?
2. Did the model confuse units or reporting boundaries?
3. How would you improve retrieval for tables and metrics pages?
4. What changes when you switch from local model to DeepSeek/OpenAI API?
5. How could you validate extracted numbers before using them in green finance analysis?
